# Life Quality Index and SWTP

This example follows the simple JCSS LQI resistance-demand calculation in [Streicher and Rackwitz (2008)](https://www.jcss-lc.org/publications/raie/07_risk_backgrounddoc_lqi_optimization.pdf): a design parameter changes the resistance, FORM estimates the failure probability, and a cost-benefit objective is checked together with an LQI/SWTP life-safety target.

The LQI helpers in Pystra are post-processing tools. They do not change the stochastic model or the reliability method; they use the `pf` and `beta` results returned by FORM, SORM, or simulation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pystra as ra

pd.options.display.float_format = "{:,.4g}".format

## Reliability model

The limit state is written as

$$g(f_y, A_s, S) = f_y A_s - 1000 S$$

where `A_s` is the design variable. For each value of `A_s`, FORM computes the failure probability and reliability index.

In [ ]:
def lsf(fy, As, S):
    return fy * As - 1000 * S


def run_reliability(As):
    limit_state = ra.LimitState(lsf)
    model = ra.StochasticModel()
    model.addVariable(ra.Lognormal("fy", 260, 18.2))
    model.addVariable(ra.Constant("As", As))
    model.addVariable(ra.Gumbel("S", 9.5, 1.5))

    options = ra.AnalysisOptions()
    options.setE1(1e-6)
    options.setE2(1e-6)

    form = ra.Form(
        analysis_options=options,
        limit_state=limit_state,
        stochastic_model=model,
    )
    form.run()
    return {"pf": float(np.atleast_1d(form.getFailure())[0]), "beta": form.getBeta()}

In [ ]:
as_values = np.array([70, 75, 80, 85.1, 90, 93, 95, 100], dtype=float)
study = ra.DesignStudy(variable="As", values=as_values, analysis=run_reliability)

reliability = study.evaluate()
reliability

## SWTP and the LQI target

Rackwitz's JCSS SWTP table is retained as a 1999 PPPUS$ anchor [Rackwitz (2008)](https://www.jcss-lc.org/publications/raie/06_risk_backgrounddoc_lqi_philosophy.pdf). For present-day studies, `indexed=True` returns an explicitly indexed value using World Bank GDP per capita PPP factors [World Bank WDI](https://data.worldbank.org/indicator/NY.GDP.PCAP.PP.CD). This keeps the literature value and the update method visible.

In [ ]:
swtp_anchor = ra.SWTP.from_country("CH")
swtp_indexed = ra.SWTP.from_country("CH", indexed=True)

pd.DataFrame(
    [
        {"basis": "Rackwitz anchor", "value_per_life": swtp_anchor.value_per_life, "price_year": swtp_anchor.price_year},
        {"basis": "GDP PPP indexed", "value_per_life": swtp_indexed.value_per_life, "price_year": swtp_indexed.price_year},
    ]
)

For the Fischer, Barnardo, and Faber target table [Fischer et al. (2012)](https://www.researchgate.net/publication/289533079_Deriving_target_reliabilities_from_the_LQI), define

$$K_1 = \frac{C_1}{\mathrm{SWTP} N_F}$$

where `C1` is the marginal safety cost and `N_F` is the expected number of fatalities conditional on failure. The result is a minimum LQI target reliability.

In [ ]:
expected_fatalities = 12
marginal_safety_cost = 5000

k1 = ra.lqi_k1(
    safety_cost_rate=marginal_safety_cost,
    swtp=swtp_indexed,
    expected_fatalities=expected_fatalities,
)
target = ra.lqi_target_reliability(k1)

pd.DataFrame([target.__dict__])

## Cost-benefit objective and LQI feasibility

The objective below follows the JCSS calculation [Streicher and Rackwitz (2008)](https://www.jcss-lc.org/publications/raie/07_risk_backgrounddoc_lqi_optimization.pdf) with a constant annual benefit, construction cost proportional to `A_s`, and failure costs discounted over the service life.

In [ ]:
costs = ra.CostBenefitModel(
    benefit_rate=1.2e4,
    interest_rate=0.02,
    service_life=100,
    construction_cost=lambda As: 5000 * As,
    failure_cost=lambda As: 5000 * As + 12 * 1.8e6 + 3e4,
)

assessment = ra.LQIAssessment(
    study=study,
    costs=costs,
    swtp=swtp_indexed,
    consequence=ra.FatalityConsequence(people_exposed=expected_fatalities),
    target=target,
)

results = assessment.evaluate()
results["construction_cost"] = 5000 * results["As"]
results["jcss_lqi_risk_cost"] = ra.jcss_lqi_risk_cost(
    results["construction_cost"],
    results["pf"],
    swtp_indexed,
    expected_fatalities,
)

results[[
    "As",
    "pf",
    "beta",
    "objective",
    "annualized_safety_cost",
    "target_pf",
    "lqi_acceptable",
    "jcss_lqi_risk_cost",
]]

In [ ]:
economic_best = results.loc[results["objective"].idxmax()]
feasible_best = results.loc[results[results["lqi_acceptable"]]["objective"].idxmax()]

pd.DataFrame(
    [
        {"selection": "economic optimum", "As": economic_best["As"], "pf": economic_best["pf"], "objective": economic_best["objective"], "lqi_acceptable": economic_best["lqi_acceptable"]},
        {"selection": "best LQI-feasible", "As": feasible_best["As"], "pf": feasible_best["pf"], "objective": feasible_best["objective"], "lqi_acceptable": feasible_best["lqi_acceptable"]},
    ]
)

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(results["As"], results["objective"], marker="o", label="objective")
ax1.set_xlabel("Reinforcement area As")
ax1.set_ylabel("Objective")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.semilogy(results["As"], results["pf"], marker="s", color="tab:red", label="failure probability")
ax2.axhline(target.pf, color="tab:red", linestyle="--", label="LQI target")
ax2.set_ylabel("Failure probability")

lines = ax1.get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc="best")
fig.tight_layout()

## Canonical JCSS marginal check

The JCSS LQI acceptance condition [Streicher and Rackwitz (2008)](https://www.jcss-lc.org/publications/raie/07_risk_backgrounddoc_lqi_optimization.pdf) can also be checked directly when a differentiable cost and failure-rate model are available:

$$\frac{dC(p)}{dp} + \mathrm{SWTP} N_F \frac{dh(p)}{dp} \ge 0$$

The helper below returns this margin. Positive values satisfy the marginal LQI condition.

In [ ]:
from scipy.stats import norm


def jcss_lognormal_pf(p):
    vr = 0.2
    vs = 0.3
    numerator = np.log(p * np.sqrt((1 + vs**2) / (1 + vr**2)))
    denominator = np.sqrt(np.log((1 + vr**2) * (1 + vs**2)))
    return norm.cdf(-numerator / denominator)


def jcss_cost(p):
    return 1e6 + 1e4 * p**1.25


pd.DataFrame(
    {
        "p": [3.0, 3.61, 4.4],
        "pf": [jcss_lognormal_pf(p) for p in [3.0, 3.61, 4.4]],
        "lqi_margin": [
            ra.jcss_lqi_acceptability(jcss_cost, jcss_lognormal_pf, p, 5e6, 10)
            for p in [3.0, 3.61, 4.4]
        ],
    }
)

## JCSS publication plots

The following cells reproduce the closed-form normal and lognormal resistance-demand curves corresponding to Figures 1 and 2 in the JCSS LQI optimization document [Streicher and Rackwitz (2008)](https://www.jcss-lc.org/publications/raie/07_risk_backgrounddoc_lqi_optimization.pdf). The publication also discusses a truncated-normal model; the executable reproduction below keeps to the closed-form equations so that every curve is transparent.

In [ ]:
from scipy.optimize import brentq, minimize_scalar


C0 = 1e6
C1_base = 1e4
cost_exponent = 1.25
damage_cost = 3 * C0
human_consequence_cost = 7e6
vr = 0.2
vs = 0.3
event_rate = 1.0
publication_swtp = 5e6
publication_expected_fatalities = 10
benefit_rate = 0.02 * C0
discount_rate = 0.0185


def publication_cost(p, C1=C1_base):
    return C0 + C1 * p**cost_exponent


def pf_publication_normal(p):
    return norm.cdf(-(p - 1) / np.sqrt((p * vr) ** 2 + vs**2))


def pf_publication_lognormal(p):
    numerator = np.log(p * np.sqrt((1 + vs**2) / (1 + vr**2)))
    denominator = np.sqrt(np.log((1 + vr**2) * (1 + vs**2)))
    return norm.cdf(-numerator / denominator)


def publication_objective(p, pf_function, C1=C1_base):
    construction = publication_cost(p, C1)
    failure_cost = construction + damage_cost + human_consequence_cost
    return (
        benefit_rate / discount_rate
        - construction
        - failure_cost * event_rate * pf_function(p) / discount_rate
    )


def publication_derivative(function, p):
    step = max(abs(p) * 1e-5, 1e-5)
    return (function(p + step) - function(p - step)) / (2 * step)


def publication_lqi_margin(p, pf_function, C1=C1_base):
    dcost = publication_derivative(lambda x: publication_cost(x, C1), p)
    drate = event_rate * publication_derivative(pf_function, p)
    return ra.jcss_lqi_acceptability_margin(
        dcost, drate, publication_swtp, publication_expected_fatalities
    )


def publication_limit(pf_function, C1=C1_base):
    return brentq(lambda p: publication_lqi_margin(p, pf_function, C1), 1.05, 10)


def publication_optimum(pf_function, C1=C1_base):
    result = minimize_scalar(
        lambda p: -publication_objective(p, pf_function, C1),
        bounds=(1.05, 10),
        method="bounded",
    )
    return result.x


publication_models = {
    "normal": pf_publication_normal,
    "lognormal": pf_publication_lognormal,
}
publication_summary = pd.DataFrame(
    [
        {
            "model": name,
            "p_opt": publication_optimum(pf_function),
            "p_lim": publication_limit(pf_function),
            "pf_opt": pf_function(publication_optimum(pf_function)),
            "pf_lim": pf_function(publication_limit(pf_function)),
        }
        for name, pf_function in publication_models.items()
    ]
)
publication_summary

In [ ]:
p_grid = np.linspace(1.2, 8.0, 220)
fig, ax = plt.subplots(figsize=(7, 4))

for name, pf_function in publication_models.items():
    objective = [publication_objective(p, pf_function) / C0 for p in p_grid]
    p_lim = publication_summary.loc[publication_summary["model"] == name, "p_lim"].item()
    ax.plot(p_grid, objective, label=f"{name} objective")
    ax.axvline(p_lim, linestyle="--", alpha=0.7, label=f"{name} LQI limit")

ax.axhline(0, color="0.5", linewidth=0.8)
ax.set_xlabel("central safety factor p")
ax.set_ylabel("objective Z(p) / C0")
ax.set_title("JCSS LQI objective curves")
ax.grid(True, alpha=0.3)
ax.legend(fontsize="small")
fig.tight_layout()

In [ ]:
cost_grid = np.logspace(3, 5, 26)
cost_effectiveness = []

for C1 in cost_grid:
    p_opt = publication_optimum(pf_publication_lognormal, C1)
    p_lim = publication_limit(pf_publication_lognormal, C1)
    cost_effectiveness.append(
        {
            "C1": C1,
            "optimal_failure_rate": event_rate * pf_publication_lognormal(p_opt),
            "acceptable_failure_rate": event_rate * pf_publication_lognormal(p_lim),
        }
    )

cost_effectiveness = pd.DataFrame(cost_effectiveness)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(
    cost_effectiveness["C1"],
    cost_effectiveness["optimal_failure_rate"],
    label="economic optimum",
)
ax.loglog(
    cost_effectiveness["C1"],
    cost_effectiveness["acceptable_failure_rate"],
    linestyle="--",
    label="LQI acceptable limit",
)
ax.set_xlabel("safety cost coefficient C1")
ax.set_ylabel("failure rate")
ax.set_title("JCSS failure rates versus safety cost")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()